In [1]:
using LowLevelFEM, LinearAlgebra

openGeometry("I-szelveny-2.geo")

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (caches not reused: 8 for different dependency version already loaded, 1 for file size changed)
Precompiling packages...
  11.8 s  ✓ LowLevelFEM
  1 dependency successfully precompiled in 14 seconds. 136 already precompiled.


In [2]:
mat = material("body")
prob = Problem([mat])

Problem("I-szelveny-2", :Solid, 3, 3, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 38194, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false)

In [3]:
n1x(x, y, z) = x
n1y(x, y, z) = y
n2x(x, y, z) = -y
n2y(x, y, z) = x
cs = CoordinateSystem([n1x, n1y, 0], [n2x, n2y, 0])
Q = rotateNodes(prob, "body", cs)

LowLevelFEM.Transformation(sparse([1, 2, 1, 2, 3, 4, 5, 4, 5, 6  …  114577, 114578, 114577, 114578, 114579, 114580, 114581, 114580, 114581, 114582], [1, 1, 2, 2, 3, 4, 4, 5, 5, 6  …  114577, 114577, 114578, 114578, 114579, 114580, 114580, 114581, 114581, 114582], [-0.4472135954999579, -0.8944271909999159, 0.8944271909999159, -0.4472135954999579, 0.9999999999999999, 0.4472135954999579, -0.8944271909999159, 0.8944271909999159, 0.4472135954999579, 0.9999999999999999  …  0.0072742420922616985, 0.999973542350988, -0.999973542350988, 0.0072742420922616985, 1.0, -0.2800380921355576, 0.9599888889737614, -0.9599888889737614, -0.2800380921355576, 0.9999999999999999], 114582, 114582), 38194, 3)

In [4]:
M0 = integrate(prob, "right", (x, y, z) -> (x^2 + y^2))

2.459885152841625e7

In [5]:
supp1 = displacementConstraint("left", uy=0)

fy(x, y, z) = √(x^2 + y^2) / M0 * 10000
load1 = load("right", fy=fy)

elsupp1 = elasticSupport("left", kz=1)
elsupp2 = elasticSupport("right", kx=1)

BoundaryCondition("right", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:kx => 1))

In [6]:
f = loadVector(prob, [load1]);

In [7]:
showDoFResults(Q * f, name="load")

0

In [8]:
K = stiffnessMatrix(prob)

sparse([1, 2, 3, 109, 110, 111, 514, 515, 516, 517  …  113238, 113356, 113357, 113358, 113737, 113738, 113739, 114580, 114581, 114582], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  114582, 114582, 114582, 114582, 114582, 114582, 114582, 114582, 114582, 114582], [348913.128032318, 132908.75886881902, 109751.91811047173, -102013.99701622548, 36128.49105146447, 43150.59589961583, 26603.875223210594, 14542.122811670886, 34442.20855667096, 31659.091883455498  …  344180.9010046459, 122764.4723799609, 68206.05412436843, -517029.6199497131, -698233.3480079278, -156666.95262796734, -1.3013082396350864e6, 816534.6109747078, -198693.6902138281, 5.41552715600475e6], 114582, 114582)

In [9]:
C1 = elasticSupportMatrix(prob, [elsupp1])

sparse([3, 111, 516, 8025, 6, 162, 165, 8028, 9, 165  …  312, 7827, 8022, 8064, 36, 513, 516, 7830, 8025, 8067], [3, 3, 3, 3, 6, 6, 6, 6, 9, 9  …  8064, 8064, 8064, 8064, 8067, 8067, 8067, 8067, 8067, 8067], [3.125534380505112, 0.8196711310637663, 0.7430960591887904, 1.562767190252558, 3.1255343805050804, 0.8196711310637594, 0.7430960591887814, 1.5627671902525422, 2.6375105162007455, 0.6608726301473206  …  1.1014973575597407, 1.157273580172595, 0.951288920697014, 5.833307131370773, 1.3187552581003765, 1.3044920148410493, 1.101497357559738, 1.1572735801725973, 0.951288920697014, 5.833307131370773], 114582, 114582)

In [10]:
C2 = elasticSupportMatrix(prob, [elsupp2])
C2 = Q * C2 * Q'

sparse([73, 74, 4705, 4706, 5029, 5030, 92446, 92447, 73, 74  …  7786, 7787, 7810, 7811, 92260, 92261, 92452, 92453, 92485, 92486], [73, 73, 73, 73, 73, 73, 73, 73, 74, 74  …  92486, 92486, 92486, 92486, 92486, 92486, 92486, 92486, 92486, 92486], [0.6588425656715194, 1.3176851313430389, 0.15297962490460537, 0.34195445566911786, 0.16859761285367258, 0.32033546442197797, 0.3184622440671995, 0.6642090838203532, 1.3176851313430389, 2.6353702626860778  …  -0.5212385720189759, 1.0424771440379514, -0.458366764901909, 0.870896853313627, -0.4257118155526531, 0.9422865280649123, -0.36926713318745, 0.7655416853757911, -2.3407743835268726, 4.656666010367819], 114582, 114582)

In [14]:
q = solveDisplacement(Q' * (K + C1 + C2) * Q, f, support=[supp1])

nodal VectorField
[0.00010429123441781427; 0.0; … ; 0.04456653174260622; -0.00145967629711366;;]

In [ ]:
#K1, f1 = applyBoundaryConditions(Q' * (K + C1 + C2) * Q, f, [supp1])

LoadError: MethodError: no method matching applyBoundaryConditions(::SystemMatrix, ::VectorField, ::Vector{BoundaryCondition})
The function `applyBoundaryConditions` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  applyBoundaryConditions([91m::Problem[39m, [91m::Vector{BoundaryCondition}[39m; steps)
[0m[90m   @[39m [35mLowLevelFEM[39m [90m~/Dokumentumok/GitHub/LowLevelFEM.jl/src/[39m[90m[4mlinear.jl:3610[24m[39m


In [ ]:
#q = (Q * K1 * Q') \ (Q * f1);

In [15]:
showDoFResults(q, :uvec, name="displacement");

In [16]:
S = solveStress(q, DoFResults=true);

In [17]:
showDoFResults(S, :syz)
showDoFResults(S, :sx)
showDoFResults(S, :sy)
showDoFResults(S, :sz)
showDoFResults(S, :sxy)
showDoFResults(S, :szx)

#plotOnPath("path", syz)

7

In [18]:
R = nodePositionVector(prob)
R = nodesToElements(R, onPhysicalGroup="left")

n2 = normalVector(prob, "left")

S2 = nodesToElements(S, onPhysicalGroup="left")

τ2 = S2 * n2;

In [19]:
showElementResults(τ2, name="shear stress")

8

In [20]:
M2 = R × (S2 * n2);

In [21]:
ez = VectorField(prob, [field("left", fx=0, fy=0, fz=1)]);

In [22]:
M2z = M2 * ez;

In [23]:
integrate(prob, "left", M2z)

1.0049700970572322e6

In [24]:
umax = probe(q, 0, 100, 1000)[1]

-7.7247e-6

In [25]:
umax / 100 / 1000 * 2e5 / 2 / (1 + 0.3) * 10^3 * (180 + 100 + 100) / 3

-0.752663076923077

In [26]:
n1 = normalVector(prob, "mid");

In [27]:
showDoFResults(n1, name="normal vector")

9

In [28]:
S2 = nodesToElements(S, onPhysicalGroup="mid")
showElementResults(S2 * n1, name="shear stress mid")

10

In [29]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()
